In [7]:
import pprint
import radiate as rd
import numpy as np
import polars as pl  # type: ignore

rd.random.seed(123)

In [8]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [9]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alter(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(
        rd.Limit.score(0.01), rd.Limit.generations(500), rd.Limit.convergence(50, 0.001)
    )
)

result = engine.run(log=True)

2026-10-17T18:13:19.224653Z  INFO State Change: PreStart -> Running
2026-10-17T18:13:19.227869Z  INFO Epoch 1    | Score:   1.4511 | Time: 2.89ms
2026-10-17T18:13:19.228896Z  INFO Epoch 2    | Score:   1.0331 | Time: 3.86ms
2026-10-17T18:13:19.229797Z  INFO Epoch 3    | Score:   1.0331 | Time: 4.74ms
2026-10-17T18:13:19.230684Z  INFO Epoch 4    | Score:   1.0297 | Time: 5.61ms
2026-10-17T18:13:19.231563Z  INFO Epoch 5    | Score:   1.0297 | Time: 6.48ms
2026-10-17T18:13:19.232435Z  INFO Epoch 6    | Score:   1.0297 | Time: 7.34ms
2026-10-17T18:13:19.233335Z  INFO Epoch 7    | Score:   1.0297 | Time: 8.21ms
2026-10-17T18:13:19.234243Z  INFO Epoch 8    | Score:   0.5113 | Time: 9.10ms
2026-10-17T18:13:19.235125Z  INFO Epoch 9    | Score:   0.5113 | Time: 9.96ms
2026-10-17T18:13:19.235974Z  INFO Epoch 10   | Score:   0.5113 | Time: 10.80ms
2026-10-17T18:13:19.236806Z  INFO Epoch 11   | Score:   0.5113 | Time: 11.62ms
2026-10-17T18:13:19.237636Z  INFO Epoch 12   | Score:   0.5113 | Time: 1

In [10]:
metrics = result.metrics
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",80.0,7380.0,80.217392,2.085144,4.347826,14.501714,80.0,100.0,92,null,null,null,null,null,null,90,1,"[""statistic""]"
"""step.evaluate.time""",0.003472,0.076466,0.00042,0.000546,2.9840e-7,0.0,8.3000e-8,0.003538,182,76466µs,420µs,546µs,0µs,3537µs,0µs,90,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,1820.0,20.0,0.0,0.0,0.0,20.0,20.0,91,null,null,null,null,null,null,90,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000004,0.000192,0.000002,0.000001,1.6003e-12,0.0,0.000001,0.000009,91,191µs,2µs,1µs,1µs,8µs,0µs,90,1,"[""selector"", ""time""]"
"""selector.boltzmann""",80.0,7280.0,80.0,0.0,0.0,0.0,80.0,80.0,91,null,null,null,null,null,null,90,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mutator.gaussian.rate""",0.3,27.300001,0.3,0.0,0.0,0.0,0.3,0.3,91,null,null,null,null,null,null,90,1,"[""alterer"", ""mutator"", … ""expr""]"
"""crossover.blend.rate""",0.7,63.700001,0.7,0.0,0.0,0.0,0.7,0.7,91,null,null,null,null,null,null,90,1,"[""alterer"", ""crossover"", … ""expr""]"
"""step.metric.time""",0.000015,0.000946,0.00001,0.000006,3.3845e-11,0.0,0.000008,0.000057,91,945µs,10µs,5µs,7µs,56µs,0µs,90,1,"[""time"", ""step""]"


In [11]:
print(result.metrics.dashboard())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["pct.carryover"].to_dict())

[33 metrics, 3004 updates]
----- Metrics ----- (33 :: 3004) 
  carryover: 0.181  diversity: 0.980  unique_members: 98  unique_scores: 98  improvements: 91  iter_time(mean): 976.39µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.280      | 0.000      | 3.000      | 100    | 0.028        | 0.653      | 0.000      | 0.000     
genome.size                | dist   | 88.000     | 88.000     | 88.000     | 100    | 8.800        | 0.000      | 0.000      | 0.000     
scores                     | dist   | 5.323      | 0.360      | 47.046     | 100    | 0.532        | 6.693      | 3.820      | 21.508    


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | Tota

In [12]:
result.duration()

datetime.timedelta(microseconds=88851)